# PYNQ-Z2 四通道迷你逻辑分析仪

本 Notebook 使用修正后的 2048 点环形缓冲设计。触发点由 `trigger_ptr` 返回，不再假定触发点固定在线性地址 `PRE_TRIGGER`。

In [ ]:
from pynq import Overlay
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time

ol = Overlay("logic_analyzer.bit")
print("IP list:", list(ol.ip_dict.keys()))

gpio_ctrl = ol.axi_gpio_0.channel1
gpio_status = ol.axi_gpio_1.channel1
bram = ol.axi_bram_ctrl_0

DEPTH = 2048
PRE_TRIGGER = 256
POST_TRIGGER = 256
TRIGGER_CH = 0
TRIGGER_MODE = 0  # 0 rising, 1 falling, 2 any edge
TEST_MODE = 1    # 1 internal test, 0 Pmod input
ctrl_idle = ((PRE_TRIGGER & 0x3FF) << 16) | \
            ((POST_TRIGGER & 0x3FF) << 6) | \
            ((TRIGGER_MODE & 0x3) << 4) | \
            ((TRIGGER_CH & 0x3) << 2) | \
            ((TEST_MODE & 0x1) << 31)
print(f"control idle = 0x{ctrl_idle:08X}")

In [ ]:
# Clear status and pointers, then arm one capture.
gpio_ctrl.write(ctrl_idle | 0x2)
time.sleep(0.01)
gpio_ctrl.write(ctrl_idle)
time.sleep(0.01)

gpio_ctrl.write(ctrl_idle | 0x1)
time.sleep(0.01)
gpio_ctrl.write(ctrl_idle)

# Wait for sticky done; do not use a fixed sleep as the completion criterion.
timeout_s = 5.0
t0 = time.time()
while True:
    status = gpio_status.read() & 0xFFFFFFFF
    if status & 0x1:
        break
    if time.time() - t0 > timeout_s:
        raise TimeoutError(f"capture timeout, status=0x{status:08X}")
    time.sleep(0.01)

triggered = (status >> 1) & 0x1
trigger_ptr = (status >> 2) & 0x7FF
print(f"done=1 triggered={triggered} trigger_ptr={trigger_ptr}")

In [ ]:
words = [bram.mmio.read(4 * i) for i in range(DEPTH)]
probes = np.array([w & 0xF for w in words], dtype=np.uint8)
timestamps = np.array([(w >> 4) & 0x0FFFFFFF for w in words], dtype=np.uint32)

start = (trigger_ptr - PRE_TRIGGER) % DEPTH
probes_ordered = np.roll(probes, -start)
timestamps_ordered = np.roll(timestamps, -start)

# PRE samples before trigger + trigger sample + POST samples after it.
valid_len = PRE_TRIGGER + POST_TRIGGER + 1
probes_valid = probes_ordered[:valid_len]
timestamps_valid = timestamps_ordered[:valid_len]

ch0 = (probes_valid >> 0) & 1
ch1 = (probes_valid >> 1) & 1
ch2 = (probes_valid >> 2) & 1
ch3 = (probes_valid >> 3) & 1

print("valid samples:", valid_len)
print("first probes:", probes_valid[:20])

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 7), sharex=True)
channels = [ch0, ch1, ch2, ch3]
for i, (ax, ch) in enumerate(zip(axes, channels)):
    ax.step(range(len(ch)), ch, where="post", linewidth=0.9)
    ax.axvline(PRE_TRIGGER, color="red", linestyle="--", alpha=0.8,
               label="Trigger" if i == 0 else None)
    ax.set_ylabel(f"CH{i}")
    ax.set_ylim(-0.2, 1.2)
    ax.grid(True)
axes[0].legend(loc="upper right")
axes[-1].set_xlabel("Sample index")
plt.suptitle("PYNQ-Z2 4-Channel Logic Analyzer")
plt.tight_layout()
plt.show()

In [ ]:
df = pd.DataFrame({
    "index": range(valid_len),
    "timestamp": timestamps_valid,
    "ch0": ch0,
    "ch1": ch1,
    "ch2": ch2,
    "ch3": ch3,
})
df.to_csv("logic_capture.csv", index=False)
print("saved logic_capture.csv")

# Optional: clear after reading so the next run starts from a known state.
gpio_ctrl.write(ctrl_idle | 0x2)
time.sleep(0.01)
gpio_ctrl.write(ctrl_idle)